#### Master Dataset Construction
Objective: This notebook merges the evolutionary history (generations, fitness scores, parameters) with the detailed execution logs (questions, answers, specific metrics).

The goal is to create a single Master CSV file that links every configuration (Hash ID) to its specific evaluation results. This allows us to analyze how specific hyperparameters influenced the quality of the RAG answers.

In [1]:
import pandas as pd

# --- 1. Load Data ---
# 'df_history': Contains the genotype (parameters) and metadata for every individual across all generations.
df_history = pd.read_csv("../results/experiments/hpo_history.csv")

# 'df_logs': Contains the detailed Q&A logs (50 questions per individual) and their specific scores.
df_logs = pd.read_csv("../results/experiments/evolution_computation_log.csv")

# Output the dimensions of the raw data
print(f"History Shape: {df_history.shape} (Individuals across all generations)")
print(f"Log Shape:     {df_logs.shape}     (Total processed questions)")

History Shape: (252, 5) (Individuals across all generations)
Log Shape:     (7600, 22)     (Total processed questions)


### 1. Consistency Check

Before merging, we verify if every individual produced by the genetic algorithm (History) has corresponding evaluation data in the logs.

We look for missing Hash IDs. A missing ID usually indicates a model failure (e.g., a timeout or a memory error during inference).

In [2]:
# --- 2. Check for Missing Data ---

# Check how many unique individuals are in the logs
unique_logs = df_logs['Hash Id'].nunique()
print(f"Unique Individuals in Logs: {unique_logs}")

# Logic: Find IDs that exist in History but NOT in Logs
# This set operation isolates the missing individuals.
missing_ids = set(df_history['Hash Id']) - set(df_logs['Hash Id'])

if len(missing_ids) > 0:
    print(f"\nWARNING: Found {len(missing_ids)} missing individuals.")
    
    # Filter the history to show exactly which configuration failed
    missing_row = df_history[df_history['Hash Id'].isin(missing_ids)]
    
    print("--- Details of Missing Individual(s) ---")
    # Transpose (.T) for better readability if there are many columns
    print(missing_row.T)
else:
    print("\nSUCCESS: All individuals have corresponding log data.")

Unique Individuals in Logs: 152

SUCCESS: All individuals have corresponding log data.


### 2. Merging Dataframes

In [3]:
# --- 3. Create Master DataFrame ---

df_full = pd.merge(
    df_history, 
    df_logs, 
    on='Hash Id', 
    how='left' # Keep all history records, even if logs are missing
)

# --- 4. Validation ---
print(f"Final Master Shape: {df_full.shape}")

# Inspect 5 random rows to verify the columns aligned correctly
print("\n--- Random Sample of Merged Data ---")
display(df_full.sample(5)) 
# Note: If you are not in Jupyter, use 'print(df_full.sample(5))' instead of 'display'

Final Master Shape: (12600, 26)

--- Random Sample of Merged Data ---


,Hash Id,gen,ind_id,fitness,params_list,chunk_size,chunk_overlap,top_k,temperature,model_name,...,f1_score,reference_contexts,retrieved_contexts,usage_metadata_question,response_metadata_question,persona_name,query_style,query_length,synthesizer_name,source_file
7585,337f89c1,3,25,0.3044,"[1, 2, 6, 9, 9]",256,25.0,6,0.9,qwen3-vl:2b-instruct,...,0.00,['Introduction\n\n...............................,Context 1: If you assembled the Original Prusa...,"{'input_tokens': 461, 'output_tokens': 76, 'to...","{'model': 'qwen3-vl:2b-instruct', 'created_at'...",Home 3D Printing Enthusiast,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,testset_prusa3d_manual_xl_105_en.csv
1655,22d8c587,0,33,0.0742,"[1, 1, 4, 3, 1]",256,12.5,4,0.3,granite4:350m,...,0.00,['1. 2. 3. 4. 5. 6.\n\nControl the printer Upd...,Context 1: installing the sheet onto the heatb...,"{'input_tokens': 233, 'output_tokens': 6, 'tot...","{'model': 'granite4:350m', 'created_at': '2026...",Office Manager,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,testset_prusa3d_manual_mini_en.csv
5078,bde14077,2,17,0.3686,"[6, 1, 8, 4, 20]",896,12.5,8,0.4,qwen3-vl:30b-a3b-instruct,...,0.43,['3.2 Disclaimer\n\nFailure to read the handbo...,Context 1: 6\n\n3.2 Disclaimer\n\nFailure to r...,"{'input_tokens': 1547, 'output_tokens': 21, 't...","{'model': 'qwen3-vl:30b-a3b-instruct', 'create...",3D Printing Enthusiast,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,testset_prusa3d_manual_mk25_en_2_51.csv
4109,996ef88f,1,40,0.3650,"[3, 2, 5, 3, 20]",512,25.0,5,0.3,qwen3-vl:30b-a3b-instruct,...,0.67,['3D PRINTING HANDBOOK\n\nFOR THE ORIGINAL PRU...,"Context 1: Tips, advice or important informati...","{'input_tokens': 585, 'output_tokens': 47, 'to...","{'model': 'qwen3-vl:30b-a3b-instruct', 'create...",3D Printing Enthusiast,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,testset_prusa3d_manual_MK35_100_en.csv
10546,ba8c8fbe,5,0,0.4148,"[6, 0, 8, 4, 20]",896,0.0,8,0.4,qwen3-vl:30b-a3b-instruct,...,0.50,"['(ABS, PLA, etc.) and filament diameter. 2.85...",Context 1: This printer supports only a 1.75 m...,"{'input_tokens': 1576, 'output_tokens': 39, 't...","{'model': 'qwen3-vl:30b-a3b-instruct', 'create...",3D Printing Enthusiast,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,testset_prusa3d_manual_mk3s_en.csv


In [4]:
# Optional: Save to disk
# df_full.to_csv("../results/experiments/master_data_combined.csv", index=False)
# print("Master CSV saved successfully.")

### 4. Integrity Check: Model Name Validation

In [5]:
import ast

# --- 1. Load and Prepare Data ---
df_run = pd.read_csv('../results/experiments/full_analysis.csv') 

# Work on a copy to preserve the original dataframe
df_clean_process = df_run.copy() 

# Clean column names (remove accidental whitespace)
df_clean_process.columns = df_clean_process.columns.str.strip()

print(f"Initial Row Count: {len(df_clean_process)}")

Initial Row Count: 12600


In [6]:
# --- 2. Validation Function ---
def is_valid_match(row):
    # 1. Target Name (from configuration column)
    target_name = str(row['model_name']).strip().lower()
    
    # 2. Actual Name (from execution log metadata)
    metadata_val = row['response_metadata_question']
    
    # Invalid if metadata is missing/NaN
    if pd.isna(metadata_val):
        return False
    
    try:
        # Parse string representation of dict to actual dict
        if isinstance(metadata_val, str):
            data = ast.literal_eval(metadata_val)
        elif isinstance(metadata_val, dict):
            data = metadata_val
        else:
            return False
            
        # Extract model name from the log
        log_name = str(data.get('model', '')).strip().lower()
        
        if not log_name:
            return False
            
        # 3. MATCH CHECK: Is one name contained in the other?
        # (Handles variations like 'qwen' vs 'qwen:instruct')
        if target_name in log_name or log_name in target_name:
            return True
        else:
            return False
            
    except Exception:
        # Catch parsing errors (malformed strings)
        return False

### Block 3: Execution & Statistics

In [7]:
# --- 3. Apply Filter ---
print("Verifying data integrity...")

# Create a boolean mask (True = Valid, False = Invalid)
mask_valid = df_clean_process.apply(is_valid_match, axis=1)

# Create the new, cleaned DataFrame
df_clean = df_clean_process[mask_valid].copy()

# --- 4. Statistics Output ---
original_count = len(df_clean_process)
clean_count = len(df_clean)
dropped_count = original_count - clean_count
loss_percentage = (dropped_count / original_count) * 100

print("-" * 40)
print(f"CLEANING COMPLETE")
print("-" * 40)
print(f"Original Rows:       {original_count}")
print(f"Valid Rows (Match):  {clean_count}")
print(f"Dropped Rows:        {dropped_count}")
print(f"Data Loss:           {loss_percentage:.2f}%")
print("-" * 40)

# Final Sanity Check: Distribution per Generation
# Ensures we still have data across all generations
print("\n--- Rows per Generation (Post-Cleaning) ---")
print(df_clean.groupby('gen')['question'].count())

Verifying data integrity...
----------------------------------------
CLEANING COMPLETE
----------------------------------------
Original Rows:       12600
Valid Rows (Match):  12600
Dropped Rows:        0
Data Loss:           0.00%
----------------------------------------

--- Rows per Generation (Post-Cleaning) ---
gen
0    2100
1    2100
2    2100
3    2100
4    2100
5    2100
Name: question, dtype: int64
